In [5]:
from __future__ import print_function, division
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------------------------
# DIRECTORIES
# --------------------------------------------------
SIM_DIR  = "/home/hp/raytrace_work/raytrace_results/Simulation_Data_201x201"
BASE_DIR = "/home/hp/raytrace_work/raytrace_results/Ray_Trajectories"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

# --------------------------------------------------
# LOAD PARAMETERS
# --------------------------------------------------
params    = np.load(os.path.join(SIM_DIR, "sim_params.npy"),
                    allow_pickle=True).item()
freq_list = np.array(params['freq_list'])
subset_A  = params['subset_A']   # 25 ray global indices
subset_B  = params['subset_B']   # 10 ray global indices
obs       = params['obs']
rsph      = params['rsph']
nx        = int(params['nx'])
ny        = int(params['ny'])
n_freqs   = len(freq_list)

# Find center ray global index explicitly
# Center ray: grid position [50, 50] --> Y=0, Z=0
trkrays_ALL = []
for i in range(nx):
    for j in range(ny):
        trkrays_ALL.append([i, j])

center_global_idx = None
for k, ray in enumerate(trkrays_ALL):
    if ray[0] == 50 and ray[1] == 50:
        center_global_idx = k
        break

print("Loaded parameters.")
print("Subset A: {} rays  Subset B: {} rays".format(
    len(subset_A), len(subset_B)))
print("Center ray [50,50] global index: {}".format(
    center_global_idx))

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def mhz_int(f):
    return int(round(f / 1.e6))

def flabel(f):
    return "{} MHz".format(mhz_int(f))

# --------------------------------------------------
# STORAGE FOR PLOT 16
# --------------------------------------------------
all_center_rays = []   # list of (label, x_array, z_array)

# ==================================================
# MAIN LOOP OVER FREQUENCIES
# ==================================================
for freq_idx, freq_hz in enumerate(freq_list):

    mhz  = mhz_int(freq_hz)
    fl   = flabel(freq_hz)
    fdir = os.path.join(BASE_DIR, "{}mhz".format(mhz))
    if not os.path.exists(fdir):
        os.makedirs(fdir)

    traj_path = os.path.join(
        SIM_DIR, "traj_{}mhz.npy".format(mhz))
    if not os.path.exists(traj_path):
        print("Missing: {}".format(traj_path))
        continue

    print("\n[{:02d}/15] {} MHz".format(freq_idx+1, mhz))
    traj = np.load(traj_path)   # shape: (10000, Nsteps, 3)
    print("  traj shape: {}".format(traj.shape))

    # Slice subsets from full trajectory array
    traj_A = traj[subset_A, :, :]   # (25, Nsteps, 3)
    traj_B = traj[subset_B, :, :]   # (10, Nsteps, 3)
    print("  traj_A: {}  traj_B: {}".format(
        traj_A.shape, traj_B.shape))

    # Collect center ray for Plot 16
    center_ray = traj[center_global_idx]   # (Nsteps, 3)
    print("  Center ray first point: "
          "X={:.4f} Y={:.4f} Z={:.4f}".format(
              center_ray[0, 0],
              center_ray[0, 1],
              center_ray[0, 2]))
    all_center_rays.append((
        fl,
        center_ray[:, 0].copy(),
        center_ray[:, 2].copy()))

    # --------------------------------------------------
    # PLOT 1: 3D trajectories — 25 rays (traj_A)
    # --------------------------------------------------
    fig  = plt.figure(figsize=(9, 7))
    ax3d = fig.add_subplot(111, projection=u'3d')

    # Sun sphere
    u_s = np.linspace(0, 2*np.pi, 30)
    v_s = np.linspace(0, np.pi, 30)
    ax3d.plot_surface(
        np.outer(np.cos(u_s), np.sin(v_s)),
        np.outer(np.sin(u_s), np.sin(v_s)),
        np.outer(np.ones(np.size(u_s)), np.cos(v_s)),
        color=u'yellow', alpha=0.5)

    colors_A = plt.cm.rainbow(
        np.linspace(0, 1, traj_A.shape[0]))
    for i in range(traj_A.shape[0]):
        ax3d.plot(traj_A[i, :, 0],
                  traj_A[i, :, 1],
                  traj_A[i, :, 2],
                  linewidth=1.2,
                  color=colors_A[i])

    ax3d.set_xlabel(u'X (Solar Radii)')
    ax3d.set_ylabel(u'Y (Solar Radii)')
    ax3d.set_zlabel(u'Z (Solar Radii)')
    ax3d.set_xlim(-rsph, rsph)
    ax3d.set_ylim(-rsph, rsph)
    ax3d.set_zlim(-rsph, rsph)
    ax3d.set_title(
        u'3D Ray Trajectories (25 rays)\n{}'.format(fl))
    plt.tight_layout()
    plt.savefig(os.path.join(
        fdir, "plot_3d_{}mhz.png".format(mhz)), dpi=150)
    plt.close()
    print("  Saved 3D plot")

    # --------------------------------------------------
    # PLOT 2: XZ plane — 10 rays (traj_B)
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 7))

    sun = plt.Circle((0.0, 0.0), 1.0,
                     fill=True, color=u'yellow', zorder=5)
    ax.add_patch(sun)
    sun_e = plt.Circle((0.0, 0.0), 1.0, fill=False,
                       edgecolor=u'orange',
                       linewidth=2, zorder=6)
    ax.add_patch(sun_e)

    for i in range(traj_B.shape[0]):
        ax.plot(traj_B[i, :, 0],
                traj_B[i, :, 2],
                color=u'blue', linewidth=1.5)

    ax.set_xlabel(u'X (Solar Radii)')
    ax.set_ylabel(u'Z (Solar Radii)')
    ax.set_title(
        u'Ray Trajectories XZ Plane (10 rays)\n{}'.format(fl))
    ax.set_xlim(-rsph, rsph)
    ax.set_ylim(-rsph, rsph)
    ax.set_aspect(u'equal')
    ax.grid(True, linestyle=u'--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(
        fdir, "plot_xz_{}mhz.png".format(mhz)), dpi=150)
    plt.close()
    print("  Saved XZ plot")

    # --------------------------------------------------
    # PLOT 3: XY plane — 25 rays (traj_A)
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 7))

    sun = plt.Circle((0.0, 0.0), 1.0,
                     fill=True, color=u'yellow', zorder=5)
    ax.add_patch(sun)
    sun_e = plt.Circle((0.0, 0.0), 1.0, fill=False,
                       edgecolor=u'orange',
                       linewidth=2, zorder=6)
    ax.add_patch(sun_e)

    for i in range(traj_A.shape[0]):
        ax.plot(traj_A[i, :, 0],
                traj_A[i, :, 1],
                color=u'green', linewidth=1.2)

    ax.set_xlabel(u'X (Solar Radii)')
    ax.set_ylabel(u'Y (Solar Radii)')
    ax.set_title(
        u'Ray Trajectories XY Plane (25 rays)\n{}'.format(fl))
    ax.set_xlim(-rsph, rsph)
    ax.set_ylim(-rsph, rsph)
    ax.set_aspect(u'equal')
    ax.grid(True, linestyle=u'--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(
        fdir, "plot_xy_{}mhz.png".format(mhz)), dpi=150)
    plt.close()
    print("  Saved XY plot")

    # --------------------------------------------------
    # PLOT 4: YZ plane — 25 rays (traj_A)
    # Shows ray positions in the image plane
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    # Sun disk cross section
    sun = plt.Circle((0.0, 0.0), 1.0,
                     fill=True, color=u'yellow', zorder=5)
    ax.add_patch(sun)
    sun_e = plt.Circle((0.0, 0.0), 1.0, fill=False,
                       edgecolor=u'orange',
                       linewidth=2, zorder=6)
    ax.add_patch(sun_e)

    for i in range(traj_A.shape[0]):
        ax.plot(traj_A[i, :, 1],   # Y
                traj_A[i, :, 2],   # Z
                linewidth=1.2,
                color=u'red')

    ax.set_xlabel(u'Y (Solar Radii)')
    ax.set_ylabel(u'Z (Solar Radii)')
    ax.set_title(
        u'Ray Trajectories YZ Plane (25 rays)\n{}'.format(fl))
    ax.set_xlim(-rsph, rsph)
    ax.set_ylim(-rsph, rsph)
    ax.set_aspect(u'equal')
    ax.grid(True, linestyle=u'--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(
        fdir, "plot_yz_{}mhz.png".format(mhz)), dpi=150)
    plt.close()
    print("  Saved YZ plot")

    # --------------------------------------------------
    # SAVE CSV — traj_A
    # --------------------------------------------------
    csv_path = os.path.join(
        fdir, "traj_A_{}mhz.csv".format(mhz))
    with open(csv_path, "wb") as f:
        f.write("ray,step,X,Y,Z\n")
        for i in range(traj_A.shape[0]):
            for s in range(traj_A.shape[1]):
                f.write("{},{},{:.6f},{:.6f},{:.6f}\n".format(
                    i, s,
                    traj_A[i, s, 0],
                    traj_A[i, s, 1],
                    traj_A[i, s, 2]))

    # --------------------------------------------------
    # SAVE CSV — traj_B
    # --------------------------------------------------
    csv_path = os.path.join(
        fdir, "traj_B_{}mhz.csv".format(mhz))
    with open(csv_path, "wb") as f:
        f.write("ray,step,X,Y,Z\n")
        for i in range(traj_B.shape[0]):
            for s in range(traj_B.shape[1]):
                f.write("{},{},{:.6f},{:.6f},{:.6f}\n".format(
                    i, s,
                    traj_B[i, s, 0],
                    traj_B[i, s, 1],
                    traj_B[i, s, 2]))

    print("  Saved CSVs to: {}".format(fdir))
    del traj, traj_A, traj_B

# ==================================================
# PLOT 16: Central ray (Y=0, Z=0) for all 15 freqs
# No Z offset — all rays start on X axis (Z=0)
# ==================================================
print("\nGenerating Plot 16...")

colors_f = plt.cm.jet(
    np.linspace(0, 1, len(all_center_rays)))

fig, ax = plt.subplots(figsize=(9, 6))

# Sun
sun = plt.Circle((0.0, 0.0), 1.0,
                 fill=True, color=u'yellow', zorder=5)
ax.add_patch(sun)
sun_e = plt.Circle((0.0, 0.0), 1.0, fill=False,
                   edgecolor=u'orange',
                   linewidth=2, zorder=6)
ax.add_patch(sun_e)

for idx, (fl, xp, zp) in enumerate(all_center_rays):
    ax.plot(xp, zp,
            color=colors_f[idx],
            linewidth=1.8,
            label=fl)

ax.set_xlabel(u'X (Solar Radii)')
ax.set_ylabel(u'Z (Solar Radii)')
ax.set_title(
    u'Central Ray (Y=0, Z=0) for All 15 Frequencies')
ax.set_xlim(-rsph, rsph)
ax.set_ylim(-5, 5)
ax.legend(fontsize=7, ncol=2, loc=u'upper right')
ax.grid(True, linestyle=u'--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(
    BASE_DIR,
    "plot16_central_ray_all_freqs.png"), dpi=200)
plt.close()
print("Saved Plot 16.")

print("\nAll done. Results: {}".format(BASE_DIR))

Loaded parameters.
Subset A: 25 rays  Subset B: 10 rays
Center ray [50,50] global index: 10100

[01/15] 15 MHz
  traj shape: (40401, 1000, 3)
  traj_A: (25, 1000, 3)  traj_B: (10, 1000, 3)
  Center ray first point: X=24.8690 Y=-0.8799 Z=-0.8799
  Saved 3D plot
  Saved XZ plot
  Saved XY plot
  Saved YZ plot
  Saved CSVs to: /home/hp/raytrace_work/raytrace_results/Ray_Trajectories/15mhz

[02/15] 30 MHz
  traj shape: (40401, 1000, 3)
  traj_A: (25, 1000, 3)  traj_B: (10, 1000, 3)
  Center ray first point: X=24.8690 Y=-0.8799 Z=-0.8799
  Saved 3D plot
  Saved XZ plot
  Saved XY plot
  Saved YZ plot
  Saved CSVs to: /home/hp/raytrace_work/raytrace_results/Ray_Trajectories/30mhz

[03/15] 45 MHz
  traj shape: (40401, 1000, 3)
  traj_A: (25, 1000, 3)  traj_B: (10, 1000, 3)
  Center ray first point: X=24.8690 Y=-0.8799 Z=-0.8799
  Saved 3D plot
  Saved XZ plot
  Saved XY plot
  Saved YZ plot
  Saved CSVs to: /home/hp/raytrace_work/raytrace_results/Ray_Trajectories/45mhz

[04/15] 60 MHz
  traj 